In [ ]:
from google.colab import drive
import pickle
import pandas as pd
from collections import defaultdict
import random
from torchvision import transforms
import torchvision.models as models
from torch.utils.data import Dataset
from PIL import Image
import os
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [ ]:
# Copying 8k to Colabs local SSD
if not os.path.exists("/content/flickr8k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr8k.zip" /content/

In [ ]:
!unzip "/content/flickr8k.zip" -d "/content/flickr8k"

Streaming output truncated to the last 5000 lines.
  inflating: /content/flickr8k/Images/3138399980_d6ab8b2272.jpg  
  inflating: /content/flickr8k/Images/3138433655_ea1d59e5b7.jpg  
  inflating: /content/flickr8k/Images/3138504165_c7ae396294.jpg  
  inflating: /content/flickr8k/Images/3138562460_44227a35cf.jpg  
  inflating: /content/flickr8k/Images/3138746531_f6b816c126.jpg  
  inflating: /content/flickr8k/Images/3139118874_599b30b116.jpg  
  inflating: /content/flickr8k/Images/3139160252_75109e9e05.jpg  
  inflating: /content/flickr8k/Images/3139238055_2817a0c7d8.jpg  
  inflating: /content/flickr8k/Images/3139389284_f01bd4c236.jpg  
  inflating: /content/flickr8k/Images/3139393607_f0a54ca46d.jpg  
  inflating: /content/flickr8k/Images/3139837262_fe5ee7ccd9.jpg  
  inflating: /content/flickr8k/Images/3139876823_859c7d7c23.jpg  
  inflating: /content/flickr8k/Images/3139895886_5a6d495b13.jpg  
  inflating: /content/flickr8k/Images/3141293960_74459f0a24.jpg  
  inflating: /content/fli

In [ ]:
print("Images:", len(os.listdir("/content/flickr8k/Images")))

Images: 8091


In [ ]:
# Copying 30k to Colabs local SSD
if not os.path.exists("/content/flickr30k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr30k.zip" /content/

In [ ]:
!unzip "/content/flickr30k.zip" -d "/content/flickr30k"

Streaming output truncated to the last 5000 lines.
  inflating: /content/flickr30k/Images/2410320522.jpg  
  inflating: /content/flickr30k/Images/2404747797.jpg  
  inflating: /content/flickr30k/Images/241046599.jpg  
  inflating: /content/flickr30k/Images/241345639.jpg  
  inflating: /content/flickr30k/Images/2409312675.jpg  
  inflating: /content/flickr30k/Images/241346471.jpg  
  inflating: /content/flickr30k/Images/2404959574.jpg  
  inflating: /content/flickr30k/Images/2410399168.jpg  
  inflating: /content/flickr30k/Images/2406591500.jpg  
  inflating: /content/flickr30k/Images/2413495734.jpg  
  inflating: /content/flickr30k/Images/241346794.jpg  
  inflating: /content/flickr30k/Images/2404488732.jpg  
  inflating: /content/flickr30k/Images/241347664.jpg  
  inflating: /content/flickr30k/Images/241345522.jpg  
  inflating: /content/flickr30k/Images/241347300.jpg  
  inflating: /content/flickr30k/Images/241347496.jpg  
  inflating: /content/flickr30k/Images/2407214681.jpg  
  inf

In [ ]:
print("Images:", len(os.listdir("/content/flickr30k/Images")))

Images: 31811


In [ ]:
DATASETS = {
    "flickr8k": {
        "ROOT": "/content/flickr8k",
        "IMAGE_DIR": "/content/flickr8k/Images",
        "CAPTION_FILE": "/content/flickr8k/captions.txt",
        "flickr_split": "/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr8k/flickr8k_split.pkl",
        "vocab":"/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr8k/vocab.pkl"
    },
    "flickr30k": {
        "ROOT": "/content/flickr30k",
        "IMAGE_DIR": "/content/flickr30k/Images",
        "CAPTION_FILE": "/content/flickr30k/captions.txt",
        "flickr_split": "/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr30k/flickr30k_split.pkl",
        "vocab":"/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr30k/vocab.pkl"
    }
}

In [ ]:
def image_caption_map(train_df,val_df,test_df):
  train_caption_map = defaultdict(list)
  val_caption_map = defaultdict(list)
  test_caption_map = defaultdict(list)

  bad_img = "861608773_bdafd5c996.jpg"

  train_caption_map.pop(bad_img, None)
  val_caption_map.pop(bad_img, None)
  test_caption_map.pop(bad_img, None)

  for _, row in train_df.iterrows():
      train_caption_map[row["image"]].append(row["caption"])

  for _, row in val_df.iterrows():
      val_caption_map[row["image"]].append(row["caption"])

  for _, row in test_df.iterrows():
      test_caption_map[row["image"]].append(row["caption"])
  return train_caption_map,val_caption_map,test_caption_map



In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Encoding

def encode_caption(text, vocab):

    tokens = text.split()

    encoded = [vocab["<SOS>"]]

    for token in tokens:
        encoded.append(
            vocab.get(token, vocab["<UNK>"])
        )

    encoded.append(vocab["<EOS>"])

    return encoded

In [ ]:
class FlickrRetrievalDataset(Dataset):
    def __init__(self, caption_map, image_dir, vocab, transform=None, random_caption=True):
        self.image_names = sorted(caption_map.keys())
        self.caption_map = caption_map
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform
        self.random_caption = random_caption

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]

        captions = self.caption_map[image_name]

        if self.random_caption:
            caption = random.choice(captions)
        else:
            caption = captions[0]      # fixed caption for eval

        try:
          image = Image.open(
              os.path.join(self.image_dir, image_name)
          ).convert("RGB")

        except Exception:
            return self.__getitem__(
                (idx + 1) % len(self)
            )

        if self.transform:
            image = self.transform(image)

        caption = torch.tensor(
            encode_caption(caption, self.vocab),
            dtype=torch.long
        )

        return image, caption

In [ ]:
class FlickrAllCaptionEvalDataset(Dataset):

    def __init__(self, caption_map, image_dir, vocab, transform=None):

        self.samples = []
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform

        for image_name in sorted(caption_map.keys()):

            for caption in caption_map[image_name]:

                self.samples.append(
                    (image_name, caption)
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        image_name, caption = self.samples[idx]

        image = Image.open(
            os.path.join(self.image_dir, image_name)
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        caption = torch.tensor(
            encode_caption(caption, self.vocab),
            dtype=torch.long
        )

        return image, caption

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    images = []
    captions = []
    lengths = []

    for image, caption in batch:
        images.append(image)
        captions.append(caption)
        lengths.append(len(caption))

    images = torch.stack(images)

    captions = pad_sequence(
        captions,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    lengths = torch.tensor(lengths)

    return images, captions, lengths

In [ ]:
def prepare_datasets_dataloaders(train_caption_map,val_caption_map,test_caption_map):

  train_dataset = FlickrRetrievalDataset(train_caption_map,IMAGE_DIR,vocab,image_transform,random_caption=True)
  val_dataset = FlickrRetrievalDataset(val_caption_map,IMAGE_DIR,vocab,image_transform,random_caption=False)
  test_dataset = FlickrRetrievalDataset(test_caption_map,IMAGE_DIR,vocab,image_transform,random_caption=False)
  all_caption_test_dataset = FlickrAllCaptionEvalDataset(test_caption_map,IMAGE_DIR,vocab,image_transform)

  BATCH_SIZE = 256
  train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True,
    )
  val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=4,
        pin_memory=True,
    )
  test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=4,
        pin_memory=True,
    )
  all_caption_test_loader = DataLoader(
    all_caption_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4
    )
  return train_loader,val_loader,test_loader,all_caption_test_loader

ResnetImage Encoder

In [ ]:
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

class ImageEncoder(nn.Module):

    def __init__(self, embed_dim=768):

        super().__init__()

        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V2
        )

        for param in backbone.parameters():
          param.requires_grad = False

        # Fine tune last two ResNet stages
        for param in backbone.layer3.parameters():
            param.requires_grad = True

        for param in backbone.layer4.parameters():
            param.requires_grad = True

        self.backbone = nn.Sequential(
            *list(backbone.children())[:-1]
        )

        self.projection = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, embed_dim)
        )

    def forward(self, images):

        features = self.backbone(images)

        features = features.squeeze(-1).squeeze(-1)

        embeddings = self.projection(features)

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

LSTM Bahdanau Attention Text encoder

In [ ]:
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)

class TextEncoderAttention(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=300,
        hidden_dim=512,
        pad_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=2,
            dropout=0.5,
            bidirectional=True,
            batch_first=True
        )

        # Attention layer
        self.attn_W = nn.Linear(hidden_dim * 2, hidden_dim)
        self.attn_U = nn.Linear(hidden_dim * 2, hidden_dim)
        self.attn_v = nn.Linear(hidden_dim, 1, bias=False)

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim * 2, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 768)
        )

    def forward(
        self,
        captions,
        lengths
    ):

        embedded = self.embedding(captions)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, (hidden, cell) = self.lstm(packed)

        outputs, _ = pad_packed_sequence(
            packed_out,
            batch_first=True
        )

        # outputs:
        # (batch, seq_len, hidden_dim*2)
        batch_size = outputs.size(0)
        seq_len = outputs.size(1)

        # Final BiLSTM hidden state (query)
        hidden_forward = hidden[-2]
        hidden_backward = hidden[-1]

        query = torch.cat(
            [hidden_forward, hidden_backward],
            dim=1
        )

        query = query.unsqueeze(1).repeat(
            1,
            seq_len,
            1
        )

        # Bahdanau Attention
        energy = torch.tanh(
            self.attn_W(outputs)
            +
            self.attn_U(query)
        )

        attn_scores = self.attn_v(energy).squeeze(-1)

        mask = (
            torch.arange(
                seq_len,
                device=outputs.device
            )
            .expand(batch_size, seq_len)
            >= lengths.unsqueeze(1).to(outputs.device)
        )

        attn_scores = attn_scores.masked_fill(
            mask,
            -1e9
        )

        attn_weights = torch.softmax(
            attn_scores,
            dim=1
        )

        context = torch.sum(
            outputs *
            attn_weights.unsqueeze(-1),
            dim=1
        )



        embeddings = self.projection(context)

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

In [ ]:
# Joint Model (ViT + LSTM Retrieval)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class ResNetLSTMRetrieval(nn.Module):
    def __init__(self, image_encoder, text_encoder):
        super().__init__()

        self.image_encoder = image_encoder
        self.text_encoder = text_encoder

        # Learnable temperature parameter
        self.logit_scale = nn.Parameter(
            torch.ones([]) * np.log(1 / 0.07)
        )

    def forward(self, images, captions, lengths):
        image_emb = self.image_encoder(images)
        text_emb = self.text_encoder(captions, lengths)

        # Normalize embeddings
        image_emb = F.normalize(image_emb, p=2, dim=1)
        text_emb = F.normalize(text_emb, p=2, dim=1)

        return image_emb, text_emb


In [ ]:
def build_image_text_encoder(vocab,device):
  encoder = ImageEncoder(embed_dim=768).to(device)

  text_encoder = TextEncoderAttention(vocab_size=len(vocab),embed_dim=300,hidden_dim=512,pad_idx=vocab["<PAD>"]).to(device)

  model = ResNetLSTMRetrieval(image_encoder=encoder,text_encoder=text_encoder).to(device)

  return model

In [ ]:
# CLIP-Style Contrastive Loss

def clip_contrastive_loss(image_emb, text_emb, logit_scale):

    # CLIP-style learnable temperature
    logit_scale = logit_scale.exp().clamp(max=100)

    # Similarity Matrix
    logits = torch.matmul(image_emb, text_emb.T) * logit_scale

    # Ground truth labels
    targets = torch.arange(
        image_emb.size(0),
        device=image_emb.device
    )

    # Image → Text
    loss_i2t = F.cross_entropy(logits, targets)

    # Text → Image
    loss_t2i = F.cross_entropy(logits.T, targets)

    # Bidirectional InfoNCE
    loss = (loss_i2t + loss_t2i) / 2

    return loss, logits

In [ ]:
# Training
def define_optimizer(model):
  optimizer = torch.optim.AdamW(
      [
          {
              "params": model.image_encoder.backbone.parameters(),
              "lr": 1e-5
          },
          {
              "params": model.image_encoder.projection.parameters(),
              "lr": 3e-4
          },
          {
              "params": model.text_encoder.parameters(),
              "lr": 3e-4
          },
          {
              "params": [model.logit_scale],
              "lr": 1e-4
          }
      ],
      weight_decay=1e-4
  )
  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=50)
  return optimizer,scheduler

In [ ]:
# Add Validation Loop
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for images, captions, lengths in dataloader:
            images = images.to(device)
            captions = captions.to(device)

            image_emb, text_emb = model(
                images,
                captions,
                lengths
            )

            loss, _ = clip_contrastive_loss(
                image_emb,
                text_emb,
                model.logit_scale
            )

            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
def model_training(model,train_loader,val_loader,optimizer,scheduler,dataset_name):
  import time
  import torch

  # Training
  NUM_EPOCHS = 50
  best_val_loss = float("inf")
  patience = 5
  epochs_without_improvement = 0

  SAVE_DIR = f"/content/drive/MyDrive/MMRetrieval/R3/{dataset_name}"
  os.makedirs(SAVE_DIR, exist_ok=True)

  BEST_MODEL_PATH = os.path.join(
      SAVE_DIR,
      "best_retrieval_R3_model.pth"
  )

  for epoch in range(NUM_EPOCHS):

      print(f"\n================ Epoch {epoch+1}/{NUM_EPOCHS} ================")

      model.train()
      running_loss = 0.0

      epoch_start = time.time()

      for batch_idx, (images, captions, lengths) in enumerate(train_loader):

          batch_start = time.time()

          images = images.to(device, non_blocking=True)
          captions = captions.to(device, non_blocking=True)

          optimizer.zero_grad(set_to_none=True)

          image_emb, text_emb = model(
              images,
              captions,
              lengths
          )

          loss, _ = clip_contrastive_loss(
              image_emb,
              text_emb,
              model.logit_scale
          )

          loss.backward()

          torch.nn.utils.clip_grad_norm_(
              model.parameters(),
              max_norm=1.0
          )

          optimizer.step()

          running_loss += loss.item()

          batch_time = time.time() - batch_start

          print(
              f"Batch {batch_idx+1:02d}/{len(train_loader)} | "
              f"Loss: {loss.item():.4f} | "
              f"Time: {batch_time:.2f}s"
          )

      train_time = time.time() - epoch_start

      train_loss = running_loss / len(train_loader)

      # ---------------- Validation ----------------
      val_start = time.time()

      val_loss = evaluate(
          model,
          val_loader,
          device
      )

      val_time = time.time() - val_start

      scheduler.step()

      print("\n---------------- Summary ----------------")
      print(f"Train Loss     : {train_loss:.4f}")
      print(f"Validation Loss: {val_loss:.4f}")
      print(f"Training Time  : {train_time:.2f} sec")
      print(f"Validation Time: {val_time:.2f} sec")
      print(
          f"GPU Memory Used: "
          f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
      )

      # Save best model
      if val_loss < best_val_loss:

          best_val_loss = val_loss
          epochs_without_improvement = 0

          torch.save(
              {
                  "model_state_dict": model.state_dict(),
                  "optimizer_state_dict": optimizer.state_dict(),
                  "epoch": epoch,
                  "scheduler_state_dict": scheduler.state_dict(),
                  "val_loss": val_loss
              },
              BEST_MODEL_PATH
          )
          print(f"✓ Best model saved(Val Loss: {val_loss:.4f})")
      else:
        epochs_without_improvement += 1
        print(f"No improvement for "f"{epochs_without_improvement}/{patience} epochs")

      if epochs_without_improvement >= patience:
        print("\nEarly stopping triggered!")
        break

  FINAL_MODEL_PATH = BEST_MODEL_PATH
  checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
  model.load_state_dict(checkpoint["model_state_dict"])

  return model,FINAL_MODEL_PATH

In [ ]:
def extract_embeddings(model, dataloader, device):
    model.eval()

    image_embeddings = []
    text_embeddings = []

    with torch.no_grad():
        for images, captions, lengths in dataloader:
            images = images.to(device)
            captions = captions.to(device)

            img_emb, txt_emb = model(images, captions, lengths)

            image_embeddings.append(img_emb.cpu())
            text_embeddings.append(txt_emb.cpu())

    image_embeddings = torch.cat(image_embeddings, dim=0)
    text_embeddings = torch.cat(text_embeddings, dim=0)

    return image_embeddings, text_embeddings

In [ ]:
# image -> text
def image_to_text_recall(similarity, k):

    correct = 0

    for img_idx in range(similarity.shape[0]):

        gt_caps = set(
            range(
                img_idx * 5,
                img_idx * 5 + 5
            )
        )

        topk = similarity[img_idx].topk(k).indices.tolist()

        if any(idx in gt_caps for idx in topk):
            correct += 1

    return correct / similarity.shape[0]


In [ ]:
def text_to_image_recall(similarity, k):

    similarity_t = similarity.T

    correct = 0

    for cap_idx in range(similarity_t.shape[0]):

        gt_img = cap_idx // 5

        topk = similarity_t[cap_idx]\
            .topk(k)\
            .indices\
            .tolist()

        if gt_img in topk:
            correct += 1

    return correct / similarity_t.shape[0]


In [ ]:
def image_to_text_mrr(similarity):

    reciprocal_ranks = []

    for img_idx in range(similarity.shape[0]):

        gt_caps = set(
            range(
                img_idx * 5,
                img_idx * 5 + 5
            )
        )

        sorted_idx = torch.argsort(
            similarity[img_idx],
            descending=True
        )

        best_rank = float("inf")

        for cap in gt_caps:

            rank = (
                (sorted_idx == cap)
                .nonzero(as_tuple=True)[0]
                .item()
            ) + 1

            best_rank = min(best_rank, rank)

        reciprocal_ranks.append(
            1.0 / best_rank
        )

    return np.mean(reciprocal_ranks)

In [ ]:
def text_to_image_mrr(similarity):

    similarity_t2i = similarity.T

    reciprocal_ranks = []

    for cap_idx in range(similarity_t2i.shape[0]):

        gt_image = cap_idx // 5

        sorted_idx = torch.argsort(
            similarity_t2i[cap_idx],
            descending=True
        )

        rank = (
            (sorted_idx == gt_image)
            .nonzero(as_tuple=True)[0]
            .item()
        ) + 1

        reciprocal_ranks.append(
            1.0 / rank
        )

    return np.mean(reciprocal_ranks)

In [ ]:
def model_testing(model,FINAL_MODEL_PATH,all_caption_test_loader):
  checkpoint = torch.load(
        FINAL_MODEL_PATH,
        map_location=device
    )
  model.load_state_dict(
        checkpoint["model_state_dict"]
    )

  model.eval()
  with torch.inference_mode():
    image_embs, text_embs = extract_embeddings(
      model,
      all_caption_test_loader,
      device
    )
  print(image_embs.shape)
  print(text_embs.shape)
  # Similarity Matrix
  unique_image_embs = image_embs[::5]

  similarity = unique_image_embs @ text_embs.T

  print("Image embeddings:", unique_image_embs.shape)
  print("Text embeddings:", text_embs.shape)
  print("Similarity:", similarity.shape)
  results = pd.DataFrame({
      "Metric": [
          "Recall@1",
          "Recall@5",
          "Recall@10",
          "MRR"
      ],
      "Image→Text": [
          image_to_text_recall(similarity, 1),
          image_to_text_recall(similarity, 5),
          image_to_text_recall(similarity, 10),
          image_to_text_mrr(similarity)
      ],
      "Text→Image": [
          text_to_image_recall(similarity, 1),
          text_to_image_recall(similarity, 5),
          text_to_image_recall(similarity, 10),
          text_to_image_mrr(similarity)
      ]
  })

  results["Image→Text"] = results["Image→Text"].round(6)
  results["Text→Image"] = results["Text→Image"].round(6)

  display(results)
  return

In [ ]:
import gc
def reset_resourses():
  gc.collect()
  torch.cuda.empty_cache()
  print("\nMemory after cleanup")
  print("Allocated:",torch.cuda.memory_allocated()/1024**3)
  print("Reserved:",torch.cuda.memory_reserved()/1024**3)

In [ ]:
for dataset_name, cfg in DATASETS.items():
    torch.cuda.reset_peak_memory_stats()
    print("\n************************")
    print(f"\nProcessing - {dataset_name}")
    print("\n************************\n")
    ROOT = cfg["ROOT"]
    IMAGE_DIR = cfg["IMAGE_DIR"]
    CAPTION_FILE = cfg["CAPTION_FILE"]
    with open(cfg["flickr_split"],"rb") as f:
      split = pickle.load(f)

    with open(cfg["vocab"],"rb") as g:
      vocab = pickle.load(g)

    model = build_image_text_encoder(vocab, device)

   # print("MODEL\n",model)

    optimizer,scheduler = define_optimizer(model)

    df = pd.read_csv(CAPTION_FILE)
    train_imgs = split["train"]
    val_imgs = split["val"]
    test_imgs = split["test"]

    print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")

    train_df = df[df["image"].isin(train_imgs)].reset_index(drop=True)
    val_df = df[df["image"].isin(val_imgs)].reset_index(drop=True)
    test_df = df[df["image"].isin(test_imgs)].reset_index(drop=True)

    train_df = train_df.dropna(subset=["caption"]).reset_index(drop=True)
    val_df = val_df.dropna(subset=["caption"]).reset_index(drop=True)
    test_df = test_df.dropna(subset=["caption"]).reset_index(drop=True)

    train_caption_map,val_caption_map,test_caption_map = image_caption_map(train_df,val_df,test_df)

    train_loader,val_loader,test_loader,all_caption_test_loader = prepare_datasets_dataloaders(train_caption_map,val_caption_map,test_caption_map)

    best_model,FINAL_MODEL_PATH  = model_training(model,train_loader,val_loader,optimizer,scheduler,dataset_name)

    model_testing(best_model,FINAL_MODEL_PATH,all_caption_test_loader)

    del model
    del optimizer
    del scheduler

    del train_loader
    del val_loader
    del test_loader
    del all_caption_test_loader

    del train_df
    del val_df
    del test_df

    del train_caption_map
    del val_caption_map
    del test_caption_map

    del split
    del vocab
    del df

    reset_resourses()


************************

Processing - flickr8k

************************

Train: 6068 | Val: 1011 | Test: 1012

================ Epoch 1/50 ================
Batch 01/24 | Loss: 5.5938 | Time: 0.22s
Batch 02/24 | Loss: 5.5949 | Time: 0.17s
Batch 03/24 | Loss: 5.5442 | Time: 0.17s
Batch 04/24 | Loss: 5.5081 | Time: 0.17s
Batch 05/24 | Loss: 5.4831 | Time: 0.17s
Batch 06/24 | Loss: 5.4093 | Time: 0.17s
Batch 07/24 | Loss: 5.3548 | Time: 0.17s
Batch 08/24 | Loss: 5.2782 | Time: 0.17s
Batch 09/24 | Loss: 5.2099 | Time: 0.17s
Batch 10/24 | Loss: 5.1567 | Time: 0.17s
Batch 11/24 | Loss: 5.1335 | Time: 0.17s
Batch 12/24 | Loss: 5.0949 | Time: 0.17s
Batch 13/24 | Loss: 5.1429 | Time: 0.20s
Batch 14/24 | Loss: 5.0840 | Time: 0.17s
Batch 15/24 | Loss: 5.0443 | Time: 0.18s
Batch 16/24 | Loss: 5.0197 | Time: 0.18s
Batch 17/24 | Loss: 4.9439 | Time: 0.19s
Batch 18/24 | Loss: 4.8775 | Time: 0.17s
Batch 19/24 | Loss: 4.8616 | Time: 0.17s
Batch 20/24 | Loss: 4.8927 | Time: 0.17s
Batch 21/24 | Loss: 4

,Metric,Image→Text,Text→Image
0,Recall@1,0.228261,0.169565
1,Recall@5,0.505929,0.428063
2,Recall@10,0.628458,0.566008
3,MRR,0.359684,0.294560



Memory after cleanup
Allocated: 0.3173823356628418
Reserved: 2.037109375

************************

Processing - flickr30k

************************

Train: 23837 | Val: 3973 | Test: 3973

================ Epoch 1/50 ================
Batch 01/94 | Loss: 5.5949 | Time: 0.20s
Batch 02/94 | Loss: 5.6376 | Time: 0.17s
Batch 03/94 | Loss: 5.5618 | Time: 0.17s
Batch 04/94 | Loss: 5.5682 | Time: 0.17s
Batch 05/94 | Loss: 5.5514 | Time: 0.20s
Batch 06/94 | Loss: 5.5181 | Time: 0.18s
Batch 07/94 | Loss: 5.5296 | Time: 0.18s
Batch 08/94 | Loss: 5.4917 | Time: 0.18s
Batch 09/94 | Loss: 5.4812 | Time: 0.21s
Batch 10/94 | Loss: 5.4780 | Time: 0.18s
Batch 11/94 | Loss: 5.4062 | Time: 0.18s
Batch 12/94 | Loss: 5.4205 | Time: 0.17s
Batch 13/94 | Loss: 5.4005 | Time: 0.21s
Batch 14/94 | Loss: 5.3570 | Time: 0.19s
Batch 15/94 | Loss: 5.2885 | Time: 0.18s
Batch 16/94 | Loss: 5.2277 | Time: 0.18s
Batch 17/94 | Loss: 5.2235 | Time: 0.17s
Batch 18/94 | Loss: 5.2163 | Time: 0.17s
Batch 19/94 | Loss: 5.1155 

,Metric,Image→Text,Text→Image
0,Recall@1,0.148251,0.113667
1,Recall@5,0.378555,0.295293
2,Recall@10,0.492827,0.403927
3,MRR,0.259886,0.206854



Memory after cleanup
Allocated: 0.3283367156982422
Reserved: 1.548828125
